In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../0_common/01. env-config"

In [0]:
%run "../0_common/04.gold-helpers"

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"
races_table = f"{catalog_name}.{silver_schema}.races"
circuits_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
races_df = spark.table(races_table).filter((F.col("batch_id") == v_batch_id))
circuits_df = spark.table(circuits_table).filter((F.col("batch_id") == v_batch_id))

In [0]:
dim_races_df = (
    races_df
        .join(circuits_df, races_df.circuit_id == circuits_df.circuit_id, "inner")
        .select(
            races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.race_date,
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country
        )
)

In [0]:
display(dim_races_df.head(5))

In [0]:
columns_to_update = [col for col in dim_races_df.columns if col not in ["season", "round"]]
columns_to_update

In [0]:
write_to_gold(
    df=dim_races_df,
    target_table=target_table,
    columns_to_update=columns_to_update,
    merge_condition="t.season = s.season AND t.round = s.round"
)